In [1]:
import sys
sys.path.append("/home/jiahuang/test-code-hazel/")
import gen_fxns
from plot_pca import plot_pca_loadings
import h5py
import numpy as np 
import matplotlib.pyplot as plt
from pathlib import Path
import scipy as sp
import sklearn
from datetime import date
import pandas as pd 
from datetime import timedelta
from scipy.stats import zscore
import seaborn as sns 
import pickle 
import os 
from datetime import datetime

In [ ]:
from scipy.signal import welch

from sklearn.decomposition import PCA


ptID = 'RCS06'
path_string = f"/userdata/jiahuang/pain-data/Stage1-test/{ptID}/biomarker/preproc_data/202605_newpreproc_all_channels/"
pt_path = f"/userdata/rvatsyayan/AnushaData/HDF5 Pain Data/{ptID}"
# electrode_df = pd.read_csv(f"/home/jiahuang/test-code-hazel/{ptID}_new_electrode_property_df_updated072026.csv")

def idx_to_channel(all_channel_labels,idx):
    match = all_channel_labels[all_channel_labels['edf_ch_idx'] == idx]
    return match['New ROI Label_all channel'].values[0]

# Function to extract numeric part from filename
def extract_number(filename):
    return int(''.join(filter(str.isdigit, filename)))
def man_z_score(array):
        array_mean = np.nanmean(array)
        array_std = np.nanstd(array)
        zscore_array = (array - array_mean)/array_std
        return zscore_array
# split trials
def split_dep(X, percent):
    idx = np.argsort(X)
    x = len(X)
    low, neutral, high = idx[:int(x*percent)], idx[int(x*percent):int(x*(1-percent))],idx[int(x*(1-percent)):]
    return high, neutral, low

def load_h5_file(path_string, pt_path, file_keyword='meanpsd_clean', dataset_name='mean_psd'):
    h5_arrays = []
    fileids = [] 
    # Get list of files in directory and sort them based on numeric part
    files = sorted(os.listdir(path_string), key=extract_number)
    
    for filename in files:
        if filename.endswith(rf'{file_keyword}.h5'):
            filepath = os.path.join(path_string, filename)
            # Load the .h5 file
            with h5py.File(filepath, 'r') as hf:
                # Load the dataset as float32
                dataset = np.array(hf[dataset_name], dtype=np.float32)
                # Append the dataset to the list
                h5_arrays.append(dataset)

    files = sorted(os.listdir(pt_path), key=extract_number)
    for filename in files:
        if filename.endswith('.h5'):
            with h5py.File(os.path.join(pt_path, filename), 'r') as hf:
                # get record_id 
                fileid_pre = (pd.DataFrame(hf['pain_info']).iloc[1,0]).decode('utf-8')
                fileid = datetime.strptime(fileid_pre, '%d-%b-%Y %H:%M:%S')
                fileids.append(fileid)
    fileids = pd.to_datetime(fileids)

    return h5_arrays, fileids


def get_missing_trial_and_channel(path_string, pt_path,ptID):
    h5_arrays, fileids = load_h5_file(path_string, pt_path)
    all_data  = []
    all_data = np.stack(h5_arrays, axis=2)
    del h5_arrays

    # %%
    file = f"/home/rvatsyayan/AnushaData/Pain_Scores_{ptID}.xlsx"
    raw_surveys = pd.read_excel(file)
    
    missing_bm_data = np.setdiff1d(raw_surveys.Timestamp, fileids)
    missing_surveys = np.setdiff1d(fileids, raw_surveys.Timestamp)
    missing_channels = [i for i in range(all_data.shape[1]) if np.any(all_data[:,i,:])!=0]
    
    idx_missing_surveys = ~np.isin(fileids, missing_surveys)

    new_alldata = all_data[:,:,idx_missing_surveys]

    idx_missing_neuraldata = ~np.isin(raw_surveys.Timestamp, missing_bm_data)
    new_surveys = raw_surveys.iloc[idx_missing_neuraldata]
    assert(new_surveys.shape[0] == new_alldata.shape[2])

    return new_surveys,idx_missing_surveys,missing_channels

def get_all_channel_labels(path_string, pt_path,ptID):
    new_surveys,idx_to_keep,missing_channels = get_missing_trial_and_channel(path_string,pt_path,ptID)
    
    electrode_df = pd.read_csv(f"/home/jiahuang/test-code-hazel/{ptID}_new_electrode_property_df_updated072026.csv")
    electrode_df["edf_ch_idx"] = (
        electrode_df["EDF Channel Number"]
        # .astype(str)
        # .str.replace(r"[^\d]", "", regex=True)
        # .astype(int)
        - 1
    )
    electrode_df_idx = [i for i in range(electrode_df.shape[0]) if str(electrode_df['New ROI Label Hazel'][i])!='nan']
    sorted_electrode_df = electrode_df.iloc[electrode_df_idx].sort_values('New ROI Label Hazel')
    save_idx_sorted = sorted_electrode_df['edf_ch_idx'].to_numpy()
    save_idx_sorted_clean = np.array([i for i in save_idx_sorted if i in missing_channels])
    electrode_df_clean = electrode_df[electrode_df["edf_ch_idx"].isin(save_idx_sorted_clean)]
    electrode_df_clean = (
        electrode_df_clean
        .set_index("edf_ch_idx")
        .loc[save_idx_sorted_clean]
        .reset_index()
    )

    all_channel_labels = pd.DataFrame({
        'New_ROI_Label_all_channel': (electrode_df_clean["New ROI Label Hazel"].astype(str) + "_" + electrode_df_clean.groupby("New ROI Label Hazel").cumcount().add(1).astype(str)).tolist(),
        'edf_ch_idx': electrode_df_clean['edf_ch_idx']
    })

    return all_channel_labels

def process_score(path_string, pt_path,ptID):
    
    new_surveys,__,__ = get_missing_trial_and_channel(path_string,pt_path,ptID)
    
    vasp = new_surveys['intensity_vas_s0'].to_numpy()
    vasd = new_surveys['mood_vas_s0'].to_numpy() # depression/mood survey scores 
    nrs = new_surveys['nrs_s0'].to_numpy()
    unpleasantness = new_surveys['unpleasantness_vas_s0'].to_numpy()
    mpq_somatic = new_surveys.iloc[:,7:18]
    mpq_affective = new_surveys[['tiring_exhausting_s0', 'sickening_s0', 'fearful_s0','punishing_cruel_s0']]
    sum_affective = np.sum(mpq_affective,axis=1).to_numpy()
    sum_somatic = np.sum(mpq_somatic,axis=1).to_numpy()

    vasp_vec = vasp.reshape(-1)
    vasd_vec = vasd.reshape(-1)
    nrs_vec = nrs.reshape(-1)
    sum_affective_vec = sum_affective.reshape(-1)
    sum_somatic_vec = sum_somatic.reshape(-1)
    unpleasantness_vec = unpleasantness.reshape(-1)

    if ptID == 'RCS08':
        mask_valid = (
            ~np.isnan(nrs_vec) &
            ~np.isnan(sum_affective_vec) &
            ~np.isnan(sum_somatic_vec) &
            ~np.isnan(unpleasantness_vec) &
            ~np.isnan(vasp_vec) &
            ((sum_affective_vec + sum_somatic_vec) != 0)
        )
    elif ptID == 'RCS09':
        mask_valid = (
            ~np.isnan(nrs_vec) &
            ~np.isnan(sum_somatic_vec) &
            ~np.isnan(unpleasantness_vec) &
            ~np.isnan(vasp_vec) &
            ((sum_affective_vec + sum_somatic_vec) != 0)
        )
    else:
        mask_valid = (
            ~np.isnan(vasd_vec) &
            ~np.isnan(nrs_vec) &
            ~np.isnan(sum_affective_vec) &
            ~np.isnan(sum_somatic_vec) &
            ~np.isnan(unpleasantness_vec) &
            ~np.isnan(vasp_vec) &
            ((sum_affective_vec + sum_somatic_vec) != 0)
        )

    vasp_clean = vasp_vec[mask_valid]
    vasd_clean = vasd_vec[mask_valid]
    nrs_clean = nrs_vec[mask_valid]
    sum_affective_clean = sum_affective_vec[mask_valid]
    sum_somatic_clean = sum_somatic_vec[mask_valid]
    unpleasantness_clean = unpleasantness_vec[mask_valid]

    vasd_z_clean = man_z_score(vasd_clean)
    nrs_z_clean = man_z_score(nrs_clean)
    affective_z_clean = man_z_score(sum_affective_clean)
    somatic_z_clean = man_z_score(sum_somatic_clean)
    unpleasant_z_clean = man_z_score(unpleasantness_clean)
    vasp_z_clean = man_z_score(vasp_clean)
    
    if ptID == 'RCS08':

        X_sensory = np.stack([
            vasp_z_clean,
            nrs_z_clean,
            somatic_z_clean
        ], axis=1)

        sensory_pca = PCA(n_components=3)
        sensory_latent = sensory_pca.fit_transform(X_sensory)

        X_affective = np.stack([
            unpleasant_z_clean,
            affective_z_clean
        ], axis=1)

        affective_pca = PCA(n_components=2)
        affective_latent = affective_pca.fit_transform(X_affective)

    elif ptID == 'RCS09':
        X_sensory = np.stack([
            vasp_z_clean,
            nrs_z_clean,
            somatic_z_clean
        ], axis=1)

        sensory_pca = PCA(n_components=3)
        sensory_latent = sensory_pca.fit_transform(X_sensory)

        X_affective = np.stack([
            unpleasant_z_clean
        ], axis=1)

        affective_latent = unpleasant_z_clean.copy()


    else:
        X_sensory = np.stack([
            vasp_z_clean,
            nrs_z_clean,
            somatic_z_clean
        ], axis=1)

        sensory_pca = PCA(n_components=3)
        sensory_latent = sensory_pca.fit_transform(X_sensory)

        X_affective = np.stack([
            -vasd_z_clean,
            unpleasant_z_clean,
            affective_z_clean
        ], axis=1)
        
        affective_pca = PCA(n_components=3)
        affective_latent = affective_pca.fit_transform(X_affective)

    if ptID !='RCS09':
        return mask_valid, affective_latent[:,0], sensory_latent[:,0],nrs_z_clean,affective_z_clean,somatic_z_clean
    else:
        return mask_valid, affective_latent, sensory_latent[:,0],nrs_z_clean,[],somatic_z_clean


def psd_summary_zscored(path_string, pt_path, ptID):
    records = []
    less10 = 0
    trial_idx = 0

    new_surveys,idx_to_keep,missing_channels = get_missing_trial_and_channel(path_string, pt_path, ptID)
    mask_valid, affective_latent, sensory_latent, nrs, mpq_aff, mpq_sense = process_score(path_string,pt_path,ptID)
    new_surveys = new_surveys.loc[mask_valid,:]
    all_channel_labels = get_all_channel_labels(path_string, pt_path, ptID)

    valid_file_indices = np.where(idx_to_keep)[0]
    valid_after_second_filter = valid_file_indices[mask_valid]
    file_to_clean = {file_idx: clean_idx for clean_idx, file_idx in enumerate(valid_after_second_filter)}


    for filename in files:
        if filename.endswith(rf'{file_keyword}.h5'):
        
            if trial_idx not in valid_after_second_filter: 
                trial_idx+=1
                continue 
            clean_idx = file_to_clean[trial_idx]

            filepath = os.path.join(path_string, filename)
            # Load the .h5 file
            with h5py.File(filepath, 'r') as hf:
                # Load the dataset as float32
                dataset = np.array(hf[dataset_name], dtype=np.float32)

                if dataset.shape[0] < 10 * srate:
                    trial_idx += 1
                    less10 += 1
                    continue
                mid = len(dataset)//2
                reref_mid = dataset[mid-5*srate: mid+5*srate]
                # Append the dataset to the list
                for row in all_channel_labels.itertuples(index=False):
                    ch_name = row.New_ROI_Label_all_channel
                    ch_idx = row.edf_ch_idx
                    # Append the dataset to the list
                    records.append({
                        'trial_idx': trial_idx,
                        'edf_ch_idx': ch_idx,
                        'ch_label': ch_name,
                        '10sec_reref':reref_mid[:,ch_idx],
                        'nrs': nrs[clean_idx],
                        'mpq_aff': mpq_aff[clean_idx],
                        # 'mpq_aff':np.nan,
                        'mpq_sense': mpq_sense[clean_idx],
                        'affective_pca': affective_latent[clean_idx],
                        'sensory_pca': sensory_latent[clean_idx]
                    })
                        
                trial_idx+=1
    print(f"Less than 10 seconds: {less10}")
    return pd.DataFrame(records)

records = load_reref_file(path_string, pt_path, ptID)
savepath = rf'/userdata/jiahuang/pain-data/Stage1-test/{ptID}/records_10secreref_granger.pickle'
records.to_pickle(savepath)

In [ ]:
# Canonical Correlation Analysis
# Step 0: Input
# X is concatenated neural data, Y is behavioral data (could be high dimensional, multiple mood metric or multiple pain metric)
from sklearn.cross_decomposition import CCA
from sklearn.preprocessing import StandardScaler

X = all_data_clean_freq_roi_z # n_trial * (len(bands)*len(large_ch)
Y = np.stack([vasd_z_clean_r, vasp_z_clean],axis=1)
# Or X = band_data_z.reshape((len(bands)*n_ch, n_trial).T -- channels not concatenated

# Step 1: Standardizing
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
Y_scaled = scaler.fit_transform(Y)

# Step 2: fitting CCA
cca = CCA(n_components=2) # could have more components
cca.fit(X_scaled, Y_scaled)
X_c, Y_c = cca.transform(X_scaled, Y_scaled)
corrs = np.corrcoef(X_c.T, Y_c.T).diagonal(offset=X_c.shape[1])

# Step 3: storing all weights
for i in range(2):
	print(cca.x_weights_[:, i])
	print(cca.y_weights_[:, i])
	
fig, ax = plt.subplots(figsize=(8,6))
plt.imshow(cca.x_weights_[:,0].reshape(len(large_ROI), len(bands)), aspect='auto')
plt.colorbar(label='Weight')
x_ticks, y_ticks = np.arange(len(bands)), np.arange(len(large_ROI))
ax.set(xticks = x_ticks, xticklabels = bands, yticks = y_ticks, yticklabels = large_ROI)
plt.title(f"{ptID} CCA1: y_depression = {np.round(cca.y_weights_[0, 0],3)}, y_pain = {np.round(cca.y_weights_[1, 0],3)}")
ax.set_xlabel("Canonical frequency")
ax.set_ylabel("ROI")
plt.show()

In [ ]:
# Regularized CCA analysis

import rcca
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X = all_data_clean_freq_roi_z # n_trial * (len(bands)*len(large_ch)
Y = np.stack([vasd_z_clean_r, vasp_z_clean],axis=1)
iterations = 200
all_numCC = []
all_numreg = []


scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X)
Y = scaler.fit_transform(Y)

for i in range(iterations):
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size=0.7)

    # from sklearn.decomposition import PCA

    # pca = PCA(n_components=0.999)
    # X_train_pca = pca.fit_transform(X_train)
    # X_test_pca  = pca.transform(X_test)

    regs = np.logspace(-3,3,10)
    ccaCV = rcca.CCACrossValidate(kernelcca=False, numCCs = [1,2],regs = regs)

    # Use the train() and validate() methods to run the analysis and perform cross-dataset prediction.
    ccaCV.train([X_train, Y_train])
    ccaCV.validate([X_test, Y_test])
    all_numCC.append(ccaCV.best_numCC)
    all_numreg.append(ccaCV.best_reg)
    
occur = [(all_numreg == i).sum() for i in regs]
best_reg = [regs[i] for i in range(len(regs)) if occur[i]==np.max(occur)][0]
print(occur, best_reg)
updated_cca = rcca.CCA(kernelcca=False, numCC = 1, reg = best_reg)
updated_cca.train([X,Y])
updated_cca.cancorrs, updated_cca.ws[1]